# Trabajo Práctico Integrador - Introducción al Análisis de datos
## Entrega 2: Limpieza, preparación y transformación de datos.


## Caso: análisis y predicción de cancelaciones en reservas hoteleras.

## 1. Datos de identificación

- **Apellido y nombre:** Rodriguez Joaquin - Pichulman Miguel
- **Comisión:** 12
- **Dataset asignado:** L
- **Entrega:** 2
- **Fecha:** 5 / 9 / 2026

# 2. Preparación del dataset


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import KBinsDiscretizer, LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler

pd.set_option("display.max_columns", None)

In [2]:
url = "https://raw.githubusercontent.com/MiguelPichulman/cuartoCuatrimestre/main/analisisDatos/integrador/hotel%20booking%20TPI%20grupo%20L.csv"

df_reservas = pd.read_csv(url)

# Verificar carga
df_reservas.head()

,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,arrival_date,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,HB-050638,City Hotel,1,302,2024,May,18,5,2024-05-05,0,2,2,0.0,0,BB,PRT,Groups,TA/TO,0,0,0,A,A,0,Non Refund,1.0,NaN,0,Transient,62.0,0,0
1,HB-042358,City Hotel,1,65,2023,September,36,5,2023-09-05,1,1,2,0.0,0,BB,PRT,Groups,TA/TO,0,0,0,A,A,0,Non Refund,1.0,NaN,0,Transient,62.0,0,0
2,HB-113042,City Hotel,0,32,2025,May,22,30,2025-05-30,0,2,2,0.0,0,SC,ISR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,160.0,0,2
3,HB-020264,Resort Hotel,0,3,2024,February,8,25,2024-02-25,0,1,1,0.0,0,BB,PRT,Online TA,TA/TO,1,0,1,A,D,0,No Deposit,240.0,NaN,0,Transient,43.0,1,2
4,HB-017627,Resort Hotel,0,5,2023,October,41,10,2023-10-10,2,6,2,0.0,0,BB,NLD,Offline TA/TO,TA/TO,0,0,0,A,A,0,No Deposit,36.0,NaN,0,Transient,42.0,0,0


# 3. Copia de trabajo

In [3]:
df_reservas_copia = df_reservas.copy()

#4. Diagnostico

In [4]:
print(df_reservas_copia.shape)

(25000, 32)


Tenemos 25000 registro y 32 columnas

Tipos de Variables

In [5]:
df_reservas_copia.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   booking_id                      25000 non-null  object 
 1   hotel                           25000 non-null  object 
 2   is_canceled                     25000 non-null  int64  
 3   lead_time                       25000 non-null  int64  
 4   arrival_date_year               25000 non-null  int64  
 5   arrival_date_month              25000 non-null  object 
 6   arrival_date_week_number        25000 non-null  int64  
 7   arrival_date_day_of_month       25000 non-null  int64  
 8   arrival_date                    25000 non-null  object 
 9   stays_in_weekend_nights         25000 non-null  int64  
 10  stays_in_week_nights            25000 non-null  int64  
 11  adults                          25000 non-null  int64  
 12  children                        

Según su clasificación estadística:

Cuantitativas discretas (conteos): 12 variables

Cualitativas nominales (categorías): 9 variables

Temporales / Fechas: 5 variables

Identificadores (claves / IDs): 3 variables

Cualitativas binarias / dicotómicas: 2 variables

Cuantitativa continua: 1 variable

Observamos que hay algunos valores nulos dentro de algunas variables

# Resumen estadistico

In [6]:
df_reservas_copia.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,24999.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,21578.000000,1385.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,0.367280,104.103800,2024.159680,26.653120,15.757160,0.934920,2.514200,1.854520,0.104524,0.007800,0.030640,0.085280,0.125720,0.222080,87.076096,185.955235,2.301840,102.401390,0.063400,0.564400
std,0.482073,106.570826,0.706458,13.340036,8.797086,1.005905,1.934577,0.513737,0.398708,0.089775,0.172344,0.852782,1.500828,0.650336,110.940284,130.025751,17.457866,58.333739,0.244505,0.786764
min,0.000000,0.000000,2023.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,9.000000,0.000000,-6.380000,0.000000,0.000000
25%,0.000000,18.000000,2024.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,70.000000,0.000000,0.000000
50%,0.000000,70.000000,2024.000000,27.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,174.000000,0.000000,95.000000,0.000000,0.000000
75%,1.000000,161.000000,2025.000000,37.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,229.000000,255.000000,0.000000,126.000000,0.000000,1.000000
max,1.000000,629.000000,2025.000000,52.000000,31.000000,16.000000,40.000000,20.000000,3.000000,2.000000,1.000000,26.000000,70.000000,21.000000,531.000000,543.000000,391.000000,5400.000000,2.000000,5.000000


In [7]:
df_reservas_copia.describe(include=['object','category'])

,booking_id,hotel,arrival_date_month,arrival_date,meal,country,market_segment,distribution_channel,reserved_room_type,assigned_room_type,deposit_type,customer_type
count,25000,25000,25000,25000,25000,24911,25000,25000,25000,25000,25000,25000
unique,25000,2,12,793,5,126,7,5,10,11,3,4
top,HB-046974,City Hotel,August,2023-12-05,BB,PRT,Online TA,TA/TO,A,A,No Deposit,Transient
freq,1,16672,3025,83,19243,10088,11917,20483,18091,15626,21948,18806


Bitacora (Creacion)

In [8]:
registros_bitacora = []

def registrar(problema, variable, decision, justificacion):
    registros_bitacora.append({
        "problema_detectado": problema,
        "variable_afectada": variable,
        "decision_tomada": decision,
        "justificacion": justificacion
    })

### Detección de valores duplicados

In [9]:
print("\n--- FILAS DUPLICADAS ---")
print(f"Cantidad de duplicados exactos: {df_reservas_copia.duplicated().sum()}")


--- FILAS DUPLICADAS ---
Cantidad de duplicados exactos: 0


### Selección de variables

In [10]:
df_reservas_copia.head()

,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,arrival_date,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,HB-050638,City Hotel,1,302,2024,May,18,5,2024-05-05,0,2,2,0.0,0,BB,PRT,Groups,TA/TO,0,0,0,A,A,0,Non Refund,1.0,NaN,0,Transient,62.0,0,0
1,HB-042358,City Hotel,1,65,2023,September,36,5,2023-09-05,1,1,2,0.0,0,BB,PRT,Groups,TA/TO,0,0,0,A,A,0,Non Refund,1.0,NaN,0,Transient,62.0,0,0
2,HB-113042,City Hotel,0,32,2025,May,22,30,2025-05-30,0,2,2,0.0,0,SC,ISR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,160.0,0,2
3,HB-020264,Resort Hotel,0,3,2024,February,8,25,2024-02-25,0,1,1,0.0,0,BB,PRT,Online TA,TA/TO,1,0,1,A,D,0,No Deposit,240.0,NaN,0,Transient,43.0,1,2
4,HB-017627,Resort Hotel,0,5,2023,October,41,10,2023-10-10,2,6,2,0.0,0,BB,NLD,Offline TA/TO,TA/TO,0,0,0,A,A,0,No Deposit,36.0,NaN,0,Transient,42.0,0,0


Analizando la variable `booking_id`,  vemos que parece representar un identificador de los huespedes.

In [11]:
len(df_reservas_copia['booking_id'].unique())

25000

Esta variable contine 25000 registros por lo que es un identificador que no tiene relacion con otras variables. Debe ser eliminada.

Busqueda de valores nulos

In [12]:
print("\n--- VALORES FALTANTES (NULOS) ---")
nulos = df_reservas_copia.isnull().sum()
display(nulos[nulos > 0])


--- VALORES FALTANTES (NULOS) ---


,0
children,1
country,89
agent,3422
company,23615


La variable `company` contiene mas del 90% de valores nulos y adamas es un identificador de la empresa asociada a la reserva. Tambien debe ser eliminada.
La variable `agent` es un identificador del agente asociado a la reserva y tambien tiene una cantdad de valores nulos, por lo que debe ser eliminada.

### Analisis de variables asociadas a `arrival_date`

Luego de analizar las variables asociadas con la fecha de llegada `arival_date`, observamos que esta incluye los datos completos de la fecha por lo que las demas varibles relacionadas  con la fecha de arrivo (`arrival_date_year`, `àrrival_date_month`, `arrival_date_week_number`, `arrival_date_day_of_month`) son redundantes. Deben ser eliminadas.

## Variables a eliminar:

In [13]:
columnas_a_descartar = [
    'booking_id',               # ID sin valor estadistico
    'company',                  # > 94% de nulos
    'agent',                     # ID sin valor estadistico
    'arrival_date_year',        # redundante con arrival_date
    'arrival_date_month',       # redundante con arrival_date
    'arrival_date_week_number', # redundante con arrival_date
    'arrival_date_day_of_month' # redundante con arrival_date
]
df_reservas_copia.drop(columns=columnas_a_descartar, inplace=True)

### Registro de Bitacora

In [14]:
registrar("Selección de variables para el análisis",
          "booking_id - company - agent - arrival_date_year - arrival_date_month - arrival_date_week_number - arrival_date_day_of_month",
          "Eliminación de las columnas",
          "Estas columnas contienen códigos de identificación, datos redundantes y  en el caso de company presenta la mayor parte de sus datos nulos, por lo que no aportan ningún tipo de información")

# Eliminacion de Registros

- limpieza de reservas nulas (adults - children - babies)

In [15]:
# Contar cuántos registros cumplen que adultos, niños y bebés = 0
cantidad = ((df_reservas_copia['adults'] == 0) &
            (df_reservas_copia['children'] == 0) &
            (df_reservas_copia['babies'] == 0)).sum()

print("Cantidad de registros con suma = 0:", cantidad)


Cantidad de registros con suma = 0: 21


### Eliminar reservas sin huespedes

In [16]:
df_reservas_copia = df_reservas_copia[(df_reservas_copia['adults'] + df_reservas_copia['children'].fillna(0) + df_reservas_copia['babies']) != 0]

### Eliminar reservas donde figuran niños sin adultos a cargo, es decir `adults==0` & (`children>0` | `babies>0`)

In [17]:
sin_adultos_original = df_reservas_copia[
    (df_reservas_copia['adults'] == 0) &
    ((df_reservas_copia['children'] > 0) | (df_reservas_copia['babies'] > 0))
]
print(len(sin_adultos_original))

50


In [18]:
df_reservas_copia = df_reservas_copia[
    ~((df_reservas_copia['adults'] == 0) &
    ((df_reservas_copia['children'] > 0) | (df_reservas_copia['babies'] > 0)))
]
print((df_reservas_copia['adults'] == 0).sum())

0


### Registro de Bitacora

In [19]:
registrar("Limpieza de registros inconsistentes",
          "adults - children - babies",
          "Eliminación de registros",
          "Se eliminan las reservas que suman 0 huéspedes en total, ya que representan transacciones sin sentido lógico o errores de carga en el sistema. Asi como auqellas en las que no figura un adulto pero si niños y bebes")

### Busqueda de valores negativos

In [23]:
negativos_por_columna = (df_reservas_copia.select_dtypes(include='number') < 0).sum()
print(negativos_por_columna)


is_canceled                       0
lead_time                         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
booking_changes                   0
days_in_waiting_list              0
adr                               0
required_car_parking_spaces       0
total_of_special_requests         0
dtype: int64


Tarifa diaria negativa (adr < 0): se encontro 1 registro con adr = -6.38 que viiola las reglas del negocio y la validez matemática; distorsiona promedios y cálculos financieros posteriores. Debe ser eliminado.

In [21]:
df_reservas_copia = df_reservas_copia[df_reservas_copia['adr'] >= 0]

In [22]:
registrar("Selección de registros para el análisis",
          "valor negativo en variable adr",
          "Eliminación de la columna",
          "Este registro contiene un valor que viola la reglas del negocio")

## Deteccion de outliers

Al analizar el resumen del dataset vemos en la variable `adr` que la diferencia de valores entre el maximo y minimo es considerable. Individualizamos el valor maximo y vemos que se corresponde con una reserva tipo A (`reserved_room_type`).

In [24]:
# Aislar el índice y mostrar el registro completo de forma vertical (.T)
indice_outlier = df_reservas_copia[df_reservas_copia['adr'] == 5400].index
df_reservas_copia.loc[indice_outlier].T

,12955
hotel,City Hotel
is_canceled,1
lead_time,35
arrival_date,2024-03-25
stays_in_weekend_nights,0
stays_in_week_nights,1
adults,2
children,0.0
babies,0
meal,BB


Buscamos todas las reservas de este tipo y vemos que es un valor que supera por mucho a las demas reservas de la misma categoria. Por lo que concluimos que estamos en presencia de un outlier que debe ser eliminado.

In [25]:
# Filtrar el DataFrame para quedarnos solo con el tipo de habitación 'A'
# y ordenarlo de mayor a menor según la tarifa diaria (adr)
df_habitacion_a = df_reservas_copia[df_reservas_copia['reserved_room_type'] == 'A'].sort_values(by='adr', ascending=False)

print(f"Cantidad de reservas con habitación tipo A: {len(df_habitacion_a)}")
df_habitacion_a.head()

Cantidad de reservas con habitación tipo A: 18074


,hotel,is_canceled,lead_time,arrival_date,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
12955,City Hotel,1,35,2024-03-25,0,1,2,0.0,0,BB,PRT,Offline TA/TO,TA/TO,0,0,0,A,A,1,Non Refund,0,Transient,5400.0,0,0
6915,Resort Hotel,0,44,2025-08-10,2,5,3,0.0,0,HB,ESP,Online TA,TA/TO,0,0,0,A,A,1,No Deposit,0,Transient-Party,337.0,0,2
18834,Resort Hotel,0,31,2024-12-30,0,2,2,0.0,0,HB,PRT,Groups,Direct,0,0,0,A,A,0,No Deposit,0,Transient-Party,305.0,0,1
1438,Resort Hotel,0,31,2024-12-30,0,2,2,0.0,0,HB,NaN,Groups,Direct,0,0,0,A,A,0,No Deposit,0,Transient-Party,305.0,0,1
9368,Resort Hotel,0,31,2024-12-30,0,2,2,0.0,0,HB,PRT,Groups,Direct,0,0,0,A,C,0,No Deposit,0,Transient-Party,305.0,0,1


In [26]:
df_reservas_copia = df_reservas_copia[(df_reservas_copia['adr'] != 5400)]

In [27]:
registrar("Selección de registros para el análisis",
          "Outlier en variable adr",
          "Eliminación del registro",
          "Luego de analizar los datos de del registro y contrastarlo con otros similares (habitacion tipo A), este registro contiene un valor que esta muy por encima del valor real por lo que demuestra ser un claro error de tipeo")

## Imputacion de valores nulos  en variables `children` y `country`

In [37]:
nulos = df_reservas_copia.isnull().sum()
display(nulos[nulos > 0])

,0
country,88


Observamos que existe un registro con la variable `children` nula. Buscamos la moda absoluta de la variable, individualizamos el registro e imputamos.



In [35]:
indice_children = df_reservas_copia[df_reservas_copia['children'].isnull()].index
df_reservas_copia.loc[indice_children].T

""
hotel
is_canceled
lead_time
arrival_date
stays_in_weekend_nights
stays_in_week_nights
adults
children
babies
meal


In [32]:
moda_children = df_reservas_copia['children'].mode()[0]

print(f"Moda absoluta de la variable children: {moda_children}")


Moda absoluta de la variable children: 0.0


In [33]:
df_reservas_copia.loc[indice_children, 'children'] = 0

In [ ]:
registrar("Selección de registros para el análisis",
          "valor negativo en variable adr",
          "Eliminación del resgistro",
          "se individualizo el registro cuyo valor de variable era nuloy se lo cambio por la moda absoluta")

#ACA QUEDAMOS!!!!!

inputacion de valores nulos en variable country por la moda de la variable

In [ ]:
# Obtener la moda
moda_country = df_reservas_copia['country'].mode()[0]

# Imputar los nulos con la moda
df_reservas_copia['country'] = df_reservas_copia['country'].fillna(moda_country)

# Verificamos
print(f"Valores imputados con la moda: {moda_country}")
print("Nulos restantes en country:", df_reservas_copia['country'].isna().sum())

Valores imputados con la moda: PRT
Nulos restantes en country: 0


## Registros con valor "undefined"

Categoria incongruente en "distribution_channel"

In [ ]:
# Reemplazar el valor 'Undefined' en distribution_channel por 'Direct'
df_reservas_copia['distribution_channel'] = df_reservas_copia['distribution_channel'].replace('Undefined', 'Direct')

# Verificación
print("Valores únicos en distribution_channel:")
print(df_reservas_copia['distribution_channel'].value_counts())

Categoria incongruente en "meal"

In [ ]:
# Reemplazar por la moda
moda_meal = df_reservas_copia['meal'].mode()[0]
df_reservas_copia['meal'] = df_reservas_copia['meal'].replace('Undefined', moda_meal)

# Verificación
print(df_reservas_copia['meal'].value_counts())

# Transformacion

## Cambio de tipo y extracción temporal a partir de arrival_date
Se convierte la variable a tipo fecha y se extraen atributos analiticos avariables temporales

In [ ]:
# Asegurar el formato datetime
df_reservas_copia['arrival_date'] = pd.to_datetime(df_reservas_copia['arrival_date'])

# Derivar mes y día de la semana para análisis temporal
df_reservas_copia['arrival_month'] = df_reservas_copia['arrival_date'].dt.month
df_reservas_copia['arrival_day_name'] = df_reservas_copia['arrival_date'].dt.day_name()

## Codificación binaria (Numerización 1 a 1 para variables dicotómicas)
City Hotel = 0, Resort Hotel = 1

In [ ]:
df_reservas_copia['hotel_encoded'] = df_reservas_copia['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

## Segmentos de anticipación (Discretización de lead_time)
Permite mitigar asimetrías y facilita el análisis por ventanas de tiempo

In [ ]:
df_reservas_copia['lead_time_segment'] = pd.cut(
    df_reservas_copia['lead_time'],
    bins=[-1, 30, 90, 180, float('inf')],
    labels=['Último momento', 'Corto plazo', 'Mediano plazo', 'Largo plazo'])

# Variables derivadas


## Cantidad total de huéspedes (adults + children + babies)
Se suman las variables para generar una métrica unificada de ocupación por habitación, útil para analizar el perfil del grupo y verificar la capacidad hotelera.

In [ ]:
df_reservas_copia['total_guests'] = (
    df_reservas_copia['adults'] +
    df_reservas_copia['children'] +
    df_reservas_copia['babies']
).astype(int)

## Duración total de la estadía
El registro original separa las noches de fin de semana (stays_in_weekend_nights) de las noches de semana (stays_in_week_nights). Esta división dificulta cuantificar el tiempo total efectivo de permanencia. Al sumarlas, se obtiene una magnitud temporal integral que permite comparar duraciones de viaje y segmentar entre estancias cortas o prolongadas.

In [ ]:
df_reservas_copia['total_stay_nights'] = (
    df_reservas_copia['stays_in_weekend_nights'] +
    df_reservas_copia['stays_in_week_nights']
)

## Importe estimado total de la estadía (duración total * tarifa diariapromedio)
La variable adr (Average Daily Rate) describe exclusivamente la tarifa diaria promedio por habitación. Multiplicar total_stay_nights por adr permite cuantificar el valor monetario global generado por cada transacción, aportando una métrica directa de ingresos para el análisis financiero y comercial

In [ ]:
df_reservas_copia['estimated_total_revenue'] = (
    df_reservas_copia['total_stay_nights'] * df_reservas_copia['adr']
)

## Reserva familiar (booleana / binaria: si viajan con niños o bebés)
Facilita la segmentación de clientes al discriminar con rapidez las reservas que involucran menores a cargo (children > 0 o babies > 0) de aquellas conformadas únicamente por adultos. Esto evita formular filtros condicionales repetitivos en las etapas posteriores de análisis exploratorio y visualización.

In [ ]:
df_reservas_copia['is_family'] = (
    (df_reservas_copia['children'] > 0) | (df_reservas_copia['babies'] > 0)
).astype(int)